In [1]:
import pandas as pd
df = pd.read_csv("../data/clean/merged_data_sets.csv")

C:\Users\franc\AppData\Local\Temp\ipykernel_24464\546622613.py:2: DtypeWarning: Columns (2: Variation) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/clean/merged_data_sets.csv")


In [2]:
df = df.drop(columns=["Unnamed: 0"])

In [3]:
df = df.drop_duplicates()

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
df_test = df[df["Variation"] == "Test"].copy()

In [6]:
df_control = df[df["Variation"] == "Control"].copy()

In [7]:
df_test = df_test[["client_id","process_step", "date_time"]]

In [8]:
df_control = df_control[["client_id","process_step", "date_time"]]

In [9]:
print("Test clients:", df_test["client_id"].nunique())
print("Control clients:", df_control["client_id"].nunique())

Test clients: 26968
Control clients: 23532


In [19]:
import numpy as np

step_rank = {"start": 0, "step_1": 1, "step_2": 2, "step_3": 3, "confirm": 4}

def clients_out_of_order(df_group, group_name):
    d = df_group.copy()
    d["timestamp"] = pd.to_datetime(d["date_time"])
    d = d.sort_values(["client_id", "timestamp"], kind="stable").reset_index(drop=True)
    d["rank"] = d["process_step"].map(step_rank)

    unknown = d.loc[d["rank"].isna(), "process_step"].unique()
    if len(unknown):
        print(f"{group_name}: unexpected step names found (ignored): {unknown}")

    g = d.groupby("client_id")
    d["event_no"] = g.cumcount() + 1
    d["prev_step"] = g["process_step"].shift(1)


    d["max_before"] = g["rank"].cummax().groupby(d["client_id"]).shift(1).fillna(-1)


    d["violation"] = d["rank"] > d["max_before"] + 1
    d["violation_type"] = np.where(d["event_no"] == 1, "wrong_start", "skipped_step")

    viol = d[d["violation"]]
    n_viol = viol.groupby("client_id").size().rename("n_violations")
    n_events = d.groupby("client_id").size().rename("n_events")

    out = (
        viol.drop_duplicates("client_id", keep="first")[
            ["client_id", "violation_type", "prev_step", "process_step", "event_no", "timestamp"]
        ]
        .rename(columns={
            "prev_step": "from_step",
            "process_step": "to_step",
            "event_no": "at_event_no",
            "timestamp": "violation_timestamp",
        })
        .join(n_viol, on="client_id")
        .join(n_events, on="client_id")
    )
    out["from_step"] = out["from_step"].fillna("(first event)")
    out.insert(0, "group", group_name)
    return out.reset_index(drop=True)

out_of_order_test = clients_out_of_order(df_test, "Test")
out_of_order_control = clients_out_of_order(df_control, "Control")

total_clients = {
    "Test": df_test["client_id"].nunique(),
    "Control": df_control["client_id"].nunique(),
}

for name, t in [("Test", out_of_order_test), ("Control", out_of_order_control)]:
    print(f"{name}: {len(t)} of {total_clients[name]} clients "
          f"({len(t) / total_clients[name] * 100:.2f}%) out of order")

Test: 419 of 26968 clients (1.55%) out of order
Control: 289 of 23532 clients (1.23%) out of order


In [18]:
out_all = pd.concat([out_of_order_test, out_of_order_control], ignore_index=True)


by_type = (
    out_all.groupby(["group", "violation_type"]).size().rename("clients").reset_index()
)
by_type["pct_of_group_clients"] = (
    by_type["clients"] / by_type["group"].map(total_clients) * 100
).round(2)


by_transition = (
    out_all.assign(transition=out_all["from_step"] + " -> " + out_all["to_step"])
    .groupby(["group", "violation_type", "transition"]).size()
    .rename("clients").reset_index()
)
by_transition["pct_of_group_clients"] = (
    by_transition["clients"] / by_transition["group"].map(total_clients) * 100
).round(2)
by_transition = by_transition.sort_values(["group", "clients"], ascending=[True, False]).reset_index(drop=True)

print("BY TYPE")
display(by_type)
print("BY TRANSITION")
display(by_transition)

print("TEST - clients out of order")
display(out_of_order_test)
print("CONTROL - clients out of order")
display(out_of_order_control)

BY TYPE


,group,violation_type,clients,pct_of_group_clients
0,Control,skipped_step,123,0.52
1,Control,wrong_start,166,0.71
2,Test,skipped_step,71,0.26
3,Test,wrong_start,348,1.29


BY TRANSITION


,group,violation_type,transition,clients,pct_of_group_clients
0,Control,wrong_start,(first event) -> step_1,92,0.39
1,Control,wrong_start,(first event) -> confirm,38,0.16
2,Control,skipped_step,step_1 -> step_3,29,0.12
3,Control,skipped_step,step_1 -> confirm,28,0.12
4,Control,wrong_start,(first event) -> step_3,28,0.12
5,Control,skipped_step,step_2 -> confirm,26,0.11
6,Control,skipped_step,start -> step_3,20,0.08
7,Control,skipped_step,start -> confirm,13,0.06
8,Control,wrong_start,(first event) -> step_2,8,0.03
9,Control,skipped_step,start -> step_2,7,0.03


TEST - clients out of order


,group,client_id,violation_type,from_step,to_step,at_event_no,violation_timestamp,n_violations,n_events
0,Test,1702,wrong_start,(first event),confirm,1,2017-04-01 11:44:04,1,3
1,Test,64635,wrong_start,(first event),confirm,1,2017-05-03 11:36:52,1,2
2,Test,77651,wrong_start,(first event),confirm,1,2017-04-26 21:58:55,1,1
3,Test,147852,wrong_start,(first event),step_3,1,2017-03-21 19:57:35,1,2
4,Test,156105,wrong_start,(first event),confirm,1,2017-03-29 23:00:15,1,18
...,...,...,...,...,...,...,...,...,...
414,Test,9895333,wrong_start,(first event),step_1,1,2017-05-15 21:41:15,1,12
415,Test,9942183,wrong_start,(first event),step_1,1,2017-04-16 12:49:56,1,4
416,Test,9953515,wrong_start,(first event),confirm,1,2017-06-02 23:26:37,1,2
417,Test,9956903,wrong_start,(first event),step_1,1,2017-03-30 08:58:48,1,13


CONTROL - clients out of order


,group,client_id,violation_type,from_step,to_step,at_event_no,violation_timestamp,n_violations,n_events
0,Control,69255,skipped_step,start,step_3,2,2017-04-05 14:44:43,1,3
1,Control,89108,wrong_start,(first event),step_2,1,2017-04-14 15:24:23,1,3
2,Control,94012,wrong_start,(first event),confirm,1,2017-05-30 17:47:29,1,2
3,Control,117320,wrong_start,(first event),confirm,1,2017-05-21 03:39:07,1,1
4,Control,193054,wrong_start,(first event),step_1,1,2017-04-18 15:35:42,1,4
...,...,...,...,...,...,...,...,...,...
284,Control,9774953,wrong_start,(first event),step_1,1,2017-05-23 23:10:43,1,4
285,Control,9808183,wrong_start,(first event),step_1,1,2017-05-24 16:39:39,1,4
286,Control,9825378,wrong_start,(first event),step_3,1,2017-04-12 14:40:11,1,2
287,Control,9871504,wrong_start,(first event),step_1,1,2017-04-09 14:47:40,1,6
